In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
import numpy as np
import tensorflow as tf

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        # Standard token embedding
        self.token_emb = tf.keras.layers.Embedding(
            input_dim=vocab_size, 
            output_dim=embed_dim
        )
        # Learned positional embedding (one vector for each position index 0..maxlen-1)
        self.pos_emb = tf.keras.layers.Embedding(
            input_dim=maxlen, 
            output_dim=embed_dim
        )
        self.maxlen = maxlen

    def call(self, x):
        seq_len = tf.shape(x)[-1]
        # Generate position indices [0, 1, 2, ..., seq_len - 1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        # Broadcast positions across batch
        positions = self.pos_emb(positions)
        tokens = self.token_emb(x)
        return tokens + positions

In [3]:
class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        
        # 1. Multi-Head Attention layer
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=d_model // num_heads
        )
        
        # 2. Point-wise Feed-Forward Network
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation="relu"),
            tf.keras.layers.Dense(d_model)
        ])
        
        # 3. Layer Normalization & Dropout
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, mask=None):
        # Self-Attention: query=inputs, value=inputs, key=inputs
        attn_output = self.mha(
            query=inputs, 
            value=inputs, 
            key=inputs, 
            attention_mask=mask
        )
        attn_output = self.dropout1(attn_output, training=training)
        # Residual connection + LayerNorm
        out1 = self.layernorm1(inputs + attn_output)
        
        # Feed-Forward network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        # Residual connection + LayerNorm
        out2 = self.layernorm2(out1 + ffn_output)
        
        return out2

In [4]:
maxlen = 8
vocab_size = 50
embed_dim = 32      # d_model
num_heads = 4
d_ff = 64
num_classes = 2

# Model definition using Functional API
inputs = tf.keras.layers.Input(shape=(maxlen,))

# 1. Embed tokens + positions
x = PositionalEmbedding(maxlen=maxlen, vocab_size=vocab_size, embed_dim=embed_dim)(inputs)

# 2. Pass through the Transformer Encoder Block
transformer_block = TransformerEncoderBlock(d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)
x = transformer_block(x)

# 3. Pool sequence representations and classify
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dropout(0.1)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 8)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ positional_embedding                 │ (None, 8, 32)               │           1,856 │
│ (PositionalEmbedding)                │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_encoder_block            │ (None, 8, 32)               │           8,544 │
│ (TransformerEncoderBlock)            │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 32)                  │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,433 (40.75 KB)

 Trainable params: 10,433 (40.75 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
sentences = [
    # Positive (1)
    "great movie loved every second of it",
    "awesome plot and brilliant acting",
    "fantastic experience highly recommended",
    "i really enjoyed this masterpiece film",
    
    # Negative (0)
    "terrible waste of time and money",
    "awful acting and completely boring plot",
    "horrible experience will never watch again",
    "bad story completely hated this film"
]

labels = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=np.float32)

In [8]:
#Vectorizer

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = vocab_size,
    output_mode = 'int',
    output_sequence_length = maxlen
)

vectorizer.adapt(sentences)

x_train = vectorizer(np.array(sentences)).numpy()

In [9]:
model.fit(
    x_train,
    labels,
    epochs=80,
    verbose=1
)

Epoch 1/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 0.5000 - loss: 0.6674
Epoch 2/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.8750 - loss: 0.5626
Epoch 3/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.6250 - loss: 0.7253
Epoch 4/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.8750 - loss: 0.4753
Epoch 5/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 1.0000 - loss: 0.4027
Epoch 6/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.8750 - loss: 0.4215
Epoch 7/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 1.0000 - loss: 0.3261
Epoch 8/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 1.0000 - loss: 0.3054
Epoch 9/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 1.0000 - loss: 0.2656
Epoch 10/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 1.0000 - loss: 0.2082
Epoch 11/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 1.0000 - loss: 0.1504
Epoch 12/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 1.0000 - loss: 0.1451
E

In [11]:
print("Transformer training complete! Loss:", model.evaluate(x_train, labels, verbose=0)[0])

Transformer training complete! Loss: 0.0011974995722994208


In [13]:
test_phrases = np.array([
    "loved the plot fantastic",    # Expected: ~1.0 (Positive)
    "terrible movie hated it"      # Expected: ~0.0 (Negative)
])

test_x = vectorizer(test_phrases).numpy()
preds = model.predict(test_x)

for phrase, prob in zip(test_phrases, preds):
    sentiment = "Positive" if prob[0] >= 0.5 else "Negative"
    print(f"Phrase: '{phrase}' -> Score: {prob[0]:.4f} ({sentiment})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Phrase: 'loved the plot fantastic' -> Score: 0.9981 (Positive)
Phrase: 'terrible movie hated it' -> Score: 0.9926 (Positive)


In [14]:
print("Vocabulary:", vectorizer.get_vocabulary())
print("Test vector:", test_x[1])

Vocabulary: ['', '[UNK]', np.str_('and'), np.str_('this'), np.str_('plot'), np.str_('of'), np.str_('film'), np.str_('experience'), np.str_('completely'), np.str_('acting'), np.str_('will'), np.str_('watch'), np.str_('waste'), np.str_('time'), np.str_('terrible'), np.str_('story'), np.str_('second'), np.str_('recommended'), np.str_('really'), np.str_('never'), np.str_('movie'), np.str_('money'), np.str_('masterpiece'), np.str_('loved'), np.str_('it'), np.str_('i'), np.str_('horrible'), np.str_('highly'), np.str_('hated'), np.str_('great'), np.str_('fantastic'), np.str_('every'), np.str_('enjoyed'), np.str_('brilliant'), np.str_('boring'), np.str_('bad'), np.str_('awful'), np.str_('awesome'), np.str_('again')]
Test vector: [14 20 28 24  0  0  0  0]


In [15]:
for word in "terrible movie hated it".split():
    idx = vectorizer([word]).numpy()[0][0]
    token_name = vectorizer.get_vocabulary()[idx] if idx < len(vectorizer.get_vocabulary()) else "[UNK]"
    print(f"'{word}' -> index {idx} ('{token_name}')")

'terrible' -> index 14 ('terrible')
'movie' -> index 20 ('movie')
'hated' -> index 28 ('hated')
'it' -> index 24 ('it')


In [16]:
test_phrases_in_vocab = np.array([
    "fantastic film loved acting",     # Expected: Positive (~1.0)
    "awful waste completely hated"      # Expected: Negative (~0.0)
])

test_x_in = vectorizer(test_phrases_in_vocab).numpy()
preds_in = model.predict(test_x_in)

for phrase, prob in zip(test_phrases_in_vocab, preds_in):
    sentiment = "Positive" if prob[0] >= 0.5 else "Negative"
    print(f"Phrase: '{phrase}' -> Score: {prob[0]:.4f} ({sentiment})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Phrase: 'fantastic film loved acting' -> Score: 0.9980 (Positive)
Phrase: 'awful waste completely hated' -> Score: 0.0031 (Negative)
